## LangchainOllama

In [32]:
from langchain_ollama import ChatOllama
import os
from dotenv import load_dotenv, find_dotenv;

load_dotenv(find_dotenv())

llm = ChatOllama(
    model=os.getenv("LOCAL_OLLAMA_MODEL"),  # type: ignore
    base_url=os.getenv("LOCAL_OLLAMA_URL"),
    temperature=0.7,
    reasoning=False
)

## Test Actual LLMs

In [33]:
llm.invoke("What is the capital of India?").content

'The capital of India is New Delhi.'

## DataSets

In [38]:
from deepeval.test_case import LLMTestCase;
from deepeval import evaluate;
from deepeval.metrics import AnswerRelevancyMetric;
from dotenv import load_dotenv, find_dotenv;
from deepeval.evaluate import AsyncConfig;
from deepeval.models import OllamaModel;
import os
from typing import Tuple, Union, Optional
from pydantic import BaseModel
from deepeval.dataset import EvaluationDataset

load_dotenv(find_dotenv())

class OllamaModelNoThink(OllamaModel):
    def generate(self, prompt: str, schema: Optional[BaseModel] = None) -> Tuple[Union[str, BaseModel], float]:
        chat_model = self.load_model()
        messages = [{"role": "user", "content": prompt}]

        response = chat_model.chat(
            model=self.name,
            messages=messages,
            format=schema.model_json_schema() if schema else None,
            options={
                **{"temperature": self.temperature},
                **self.generation_kwargs,
            },
            think=False
        )
        return (
            (
                schema.model_validate_json(response.message.content)
                if schema
                else response.message.content
            ),
            0,
        )
    
    async def a_generate(self, prompt: str, schema: Optional[BaseModel] = None) -> Tuple[Union[str, BaseModel], float]:
        chat_model = self.load_model(async_mode=True)
        messages = [{"role": "user", "content": prompt}]

        response = await chat_model.chat(
            model=self.name,
            messages=messages,
            format=schema.model_json_schema() if schema else None,
            options={
                **{"temperature": self.temperature},
                **self.generation_kwargs,
            },
            think=False
        )
        return (
            (
                schema.model_validate_json(response.message.content)
                if schema
                else response.message.content
            ),
            0,
        )

ollama_model = OllamaModelNoThink(model=os.getenv("LOCAL_OLLAMA_MODEL"), base_url=os.getenv("LOCAL_OLLAMA_URL"))

test_case_1 = LLMTestCase(
    input="What is the capital of India?",
    expected_output="The capital of India is New Delhi.",
    actual_output=llm.invoke("What is the capital of India?").content # type: ignore
)

test_case_2 = LLMTestCase(
    input="Where is the India Gate?",
    expected_output="New Delhi",
    actual_output=llm.invoke("Where is the India Gate?").content # type: ignore
)
dataset = EvaluationDataset()

dataset.add_test_case(test_case_1)
dataset.add_test_case(test_case_2)


evaluate(test_cases =dataset.test_cases, 
         metrics=[AnswerRelevancyMetric(model=ollama_model)],
         async_config=AsyncConfig(run_async=False)
)

✨ You're running DeepEval's latest Answer Relevancy Metric! (using deepseek-r1:1.5b (Ollama), strict=False, 
async_mode=False)...

c:\Users\SANexGenUser\Desktop\USA_Testing\AI\deepeval-llm-evaluation\.venv\Lib\site-packages\rich\live.py:260: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Metrics Summary

  - ✅ Answer Relevancy (score: 0.5, threshold: 0.5, strict: False, evaluation model: deepseek-r1:1.5b (Ollama), reason: The answer is correct, but it's a bit too straightforward., error: None)

For test case:

  - input: What is the capital of India?
  - actual output: The capital of India is Delhi.
  - expected output: The capital of India is New Delhi.
  - context: None
  - retrieval context: None


Metrics Summary

  - ✅ Answer Relevancy (score: 0.6666666666666666, threshold: 0.5, strict: False, evaluation model: deepseek-r1:1.5b (Ollama), reason: The answer relevance score of 0.67 indicates that there are some relevant points, but they lack sufficient context to fully address the question about where the India Gate is located., error: None)

For test case:

  - input: Where is the India Gate?
  - actual output: The India Gate is located on the bank of the river Ganges in the city of Kolkata, which is part of West Bengal. It is a symbol of India's rich history and

⚠ WARNING: No hyperparameters logged.
» ]8;id=2097407;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 118.65s | token cost: None)
» Test Results (2 total tests):
   » Pass Rate: 100.0% | Passed: 2 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Answer Relevancy', threshold=0.5, success=True, score=0.5, reason="The answer is correct, but it's a bit too straightforward.", strict_mode=False, evaluation_model='deepseek-r1:1.5b (Ollama)', error=None, evaluation_cost=0.0, verbose_logs='Statements:\n[\n    "The capital of India is Delhi.",\n    "Delhi is the capital city of India."\n] \n \nVerdicts:\n[\n    {\n        "verdict": "yes",\n        "reason": "The statement directly addresses the question about the capital of India."\n    },\n    {\n        "verdict": "no",\n        "reason": "The statements are factual and do not require interpretation."\n    }\n]')], conversational=False, multimodal=False, input='What is the capital of India?', actual_output='The capital of India is Delhi.', expected_output='The capital of India is New Delhi.', context=None, retrieval_context=None, turns=None, additional_metadata=None), TestResult

## Creating Goldens 

In [10]:
from deepeval.dataset import EvaluationDataset, Golden

golden = Golden(
    input="What is the capital of India?",
    expected_output="The capital of India is New Delhi.",
    context=["India Gate is a famous war memorial located in the capital city of India. It was built to honor the soldiers of the British Indian Army who lost their lives during World War I. The monument is one of the most popular tourist attractions in the country and is visited by thousands of people every day. It is situated near Rajpath and is surrounded by important government buildings and public spaces. Many national celebrations and public gatherings take place around this monument throughout the year. India Gate is located in New Delhi, which is a part of the National Capital Territory of Delhi in India."]
) # type: ignore

dataset = EvaluationDataset()
dataset.add_golden(golden)

In [11]:
dataset

EvaluationDataset(test_cases=[], goldens=[Golden(input='What is the capital of India?', actual_output=None, expected_output='The capital of India is New Delhi.', context=['India Gate is a famous war memorial located in the capital city of India. It was built to honor the soldiers of the British Indian Army who lost their lives during World War I. The monument is one of the most popular tourist attractions in the country and is visited by thousands of people every day. It is situated near Rajpath and is surrounded by important government buildings and public spaces. Many national celebrations and public gatherings take place around this monument throughout the year. India Gate is located in New Delhi, which is a part of the National Capital Territory of Delhi in India.'], retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None, multimodal=False, images_mapping=None)], _alias=None,

In [13]:
test_data = [
    {
        "input": "What is the capital of India?",
        "expected_output": "The capital of India is New Delhi."
    },
    {
        "input": "Where is the India Gate?",
        "expected_output": "New Delhi"
    }
]

In [15]:
goldens = []

for data in test_data:
    golden = Golden(
        input=data["input"],
        expected_output=data["expected_output"],
        context=["India Gate is a famous war memorial located in the capital city of India. It was built to honor the soldiers of the British Indian Army who lost their lives during World War I. The monument is one of the most popular tourist attractions in the country and is visited by thousands of people every day. It is situated near Rajpath and is surrounded by important government buildings and public spaces. Many national celebrations and public gatherings take place around this monument throughout the year. India Gate is located in New Delhi, which is a part of the National Capital Territory of Delhi in India."]
    ) # type: ignore

    goldens.append(golden)

dataset = EvaluationDataset(goldens=goldens)
dataset

EvaluationDataset(test_cases=[], goldens=[Golden(input='What is the capital of India?', actual_output=None, expected_output='The capital of India is New Delhi.', context=['India Gate is a famous war memorial located in the capital city of India. It was built to honor the soldiers of the British Indian Army who lost their lives during World War I. The monument is one of the most popular tourist attractions in the country and is visited by thousands of people every day. It is situated near Rajpath and is surrounded by important government buildings and public spaces. Many national celebrations and public gatherings take place around this monument throughout the year. India Gate is located in New Delhi, which is a part of the National Capital Territory of Delhi in India.'], retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None, multimodal=False, images_mapping=None), Golden(input=

## Creating a Large Adversarial Golden Dataset

In [42]:
import json


dataset = EvaluationDataset()

with open('../dev.json', 'r') as f:
    data = json.load(f)

for article in data['data']:
    for para in article['paragraphs']:
        context = para['context']
        for qa in para['qas']:
            expected_output = qa['answers'][0]['text'] if qa['answers'] else None
            dataset.add_golden(Golden(
                input=qa['question'],
                expected_output=expected_output,
                context=[context]
            )) # type: ignore

    # print(f"Loaded {len(context)} context from dev.json")
print(f"Loaded {len(dataset.goldens)} golden from dev.json")


Loaded 1000 golden from dev.json


In [ ]:
dataset.goldens[0]

Golden(input='Which environment has more precipitation, the rainforest or the savanna?', actual_output=None, expected_output='rainforest', context=['Following the Cretaceous–Paleogene extinction event, the extinction of the dinosaurs and the wetter climate may have allowed the tropical rainforest to spread out across the continent. From 66–34 Mya, the rainforest extended as far south as 45°. Climate fluctuations during the last 34 million years have allowed savanna regions to expand into the tropics. During the Oligocene, for example, the rainforest spanned a relatively narrow band. It expanded again during the Middle Miocene, then retracted to a mostly inland formation at the last glacial maximum. However, the rainforest still managed to thrive during these glacial periods, allowing for the survival and evolution of a broad diversity of species.'], retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, cust

## Convert Goldens to LLMTestCase

In [43]:
from langchain_core.messages import SystemMessage, HumanMessage

dataset.test_cases.clear()

for golden in dataset.goldens[:5]:

    context_text = "\n\n".join(golden.context) if golden.context else ""
    messages = [
        SystemMessage(content=f"Use the following context to answer the question:\n\n{context_text}"),
        HumanMessage(content=golden.input) # type: ignore
    ]

    testcase = LLMTestCase(
        input=golden.input, # type: ignore
        expected_output=golden.expected_output, # type: ignore
        context=golden.context,
        actual_output=llm.invoke(messages).content # type: ignore
        )
    
    dataset.add_test_case(testcase)

dataset.test_cases

[LLMTestCase(input='What is teaching not considered due to stress?', actual_output='The question does not provide specific information about how "teaching" is considered in the absence of stress, so it cannot be answered based on the given context.', expected_output='average profession', context=['A 2000 study found that 42% of UK teachers experienced occupational stress, twice the figure for the average profession. A 2012 study found that teachers experienced double the rate of anxiety, depression, and stress than average workers.'], retrieval_context=None, additional_metadata=None, tools_called=None, comments=None, expected_tools=None, token_cost=None, completion_time=None, multimodal=False, name=None, tags=None, mcp_servers=None, mcp_tools_called=None, mcp_resources_called=None, mcp_prompts_called=None, custom_column_key_values=None),
 LLMTestCase(input='What do normal workers not have to deal with as much?', actual_output='In China, we firmly believe that all citizens are equal bef

In [44]:
evaluate(test_cases = dataset.test_cases,
         metrics=[AnswerRelevancyMetric(model=ollama_model)],
         async_config=AsyncConfig(run_async=False)
)

✨ You're running DeepEval's latest Answer Relevancy Metric! (using deepseek-r1:1.5b (Ollama), strict=False, 
async_mode=False)...

c:\Users\SANexGenUser\Desktop\USA_Testing\AI\deepeval-llm-evaluation\.venv\Lib\site-packages\rich\live.py:260: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Metrics Summary

  - ❌ Answer Relevancy (score: 0.0, threshold: 0.5, strict: False, evaluation model: deepseek-r1:1.5b (Ollama), reason: The statement does not provide sufficient context to determine whether teaching is considered in the absence of stress., error: None)

For test case:

  - input: What is teaching not considered due to stress?
  - actual output: The question does not provide specific information about how "teaching" is considered in the absence of stress, so it cannot be answered based on the given context.
  - expected output: average profession
  - context: ['A 2000 study found that 42% of UK teachers experienced occupational stress, twice the figure for the average profession. A 2012 study found that teachers experienced double the rate of anxiety, depression, and stress than average workers.']
  - retrieval context: None


Metrics Summary

  - ✅ Answer Relevancy (score: 0.5, threshold: 0.5, strict: False, evaluation model: deepseek-r1:1.5b (Ollama), reason: The a

⚠ WARNING: No hyperparameters logged.
» ]8;id=2097411;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 326.05s | token cost: None)
» Test Results (5 total tests):
   » Pass Rate: 80.0% | Passed: 4 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=False, metrics_data=[MetricData(name='Answer Relevancy', threshold=0.5, success=False, score=0.0, reason='The statement does not provide sufficient context to determine whether teaching is considered in the absence of stress.', strict_mode=False, evaluation_model='deepseek-r1:1.5b (Ollama)', error=None, evaluation_cost=0.0, verbose_logs='Statements:\n[\n    "The question does not provide specific information about how \'teaching\' is considered in the absence of stress."\n] \n \nVerdicts:\n[\n    {\n        "verdict": "no",\n        "reason": "The statement does not provide sufficient context to determine whether teaching is considered in the absence of stress."\n    }\n]')], conversational=False, multimodal=False, input='What is teaching not considered due to stress?', actual_output='The question does not provide specific information about how "teaching" is considered in the absence of stress, so it cannot be answer